# Full Pipeline

## Imports

In [2]:
import os
import cv2
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
import segmentation_models_pytorch as smp
import torchvision.transforms.v2 as T
from collections import defaultdict

pl.seed_everything(23)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MEAN = torch.tensor([0.485, 0.456, 0.406])
STD = torch.tensor([0.229, 0.224, 0.225])

def denormalize(tensor):
    m = MEAN.view(3, 1, 1).to(tensor.device)
    s = STD.view(3, 1, 1).to(tensor.device)
    return torch.clamp(tensor * s + m, 0, 1)

Seed set to 23


## Models

In [4]:
class FenceSegmentationLightning(pl.LightningModule):
    def __init__(self, learning_rate=3e-4):
        super().__init__()
        self.model = smp.Unet(encoder_name="resnet50", encoder_weights="imagenet", in_channels=3, classes=1)

    def forward(self, x):
        return self.model(x)

class LightningModuleInpainting(pl.LightningModule):
    def __init__(self, learning_rate=3e-4):
        super().__init__()
        # Entrada de 4 canales: RGB (3) + Mask (1)
        self.model = smp.Unet(encoder_name='resnet50', encoder_weights='imagenet', in_channels=4, classes=3)
        
    def forward(self, x):
        return self.model(x)

## Pipeline

In [ ]:
class FenceRemovalPipeline:
    def __init__(self, seg_ckpt, inp_ckpt):
        self.seg_model = FenceSegmentationLightning.load_from_checkpoint(seg_ckpt).to(DEVICE).eval()
        self.inp_model = LightningModuleInpainting.load_from_checkpoint(inp_ckpt).to(DEVICE).eval()

    def process(self, image_tensor, dilation_k=5, patch_size=224, halo_size=224):
        """
        Processes the image using a patch-based approach with context halo.
        
        Args:
            image_tensor: [1, 3, H, W] normalized
            patch_size: Size of the core patch to predict (e.g., 224)
            halo_size: Extra context around each patch (e.g., 224)
            
        The total extracted region will be (patch_size + 2*halo_size) x (patch_size + 2*halo_size),
        which gets resized down to 224x224 for the model, then resized back up.
        """
        _, _, H, W = image_tensor.shape
        
        # Adapt halo size to image dimensions (can't pad more than the dimension)
        # Leave at least 1 pixel for padding to work
        max_halo_h = max(1, H - 1)
        max_halo_w = max(1, W - 1)
        effective_halo = min(halo_size, max_halo_h, max_halo_w)
        
        # If image is smaller than patch_size, adjust patch_size too
        effective_patch = min(patch_size, H, W)
        stride = effective_patch  # Non-overlapping core patches
        
        # Pad image to handle edges
        img_padded = F.pad(image_tensor, (effective_halo, effective_halo, effective_halo, effective_halo), mode='reflect')
        
        full_raw_mask = torch.zeros((1, 1, H, W), device=DEVICE)
        count_mask = torch.zeros((1, 1, H, W), device=DEVICE)

        with torch.no_grad():
            # Step 1: Patch-based Segmentation with Context Halo
            for i in range(0, H, stride):
                for j in range(0, W, stride):
                    # Ensure we don't go out of bounds
                    i_safe = min(i, H - effective_patch)
                    j_safe = min(j, W - effective_patch)
                    
                    # Extract large context patch from padded image
                    # In padded coords, original (0,0) is at (effective_halo, effective_halo)
                    y_start = i_safe
                    x_start = j_safe
                    y_end = i_safe + effective_patch + (2 * effective_halo)
                    x_end = j_safe + effective_patch + (2 * effective_halo)
                    
                    large_patch = img_padded[:, :, y_start:y_end, x_start:x_end]
                    
                    # Resize to model input size (224x224)
                    input_patch = T.Resize((224, 224), antialias=True)(large_patch)
                    
                    # Predict mask
                    seg_logits = self.seg_model(input_patch)
                    probs = torch.sigmoid(seg_logits)
                    
                    # Resize mask back to context size
                    context_size = effective_patch + 2 * effective_halo
                    mask_large = T.Resize((context_size, context_size), antialias=True)(probs)
                    
                    # Crop out center (removing halo)
                    center_mask = mask_large[:, :, effective_halo:effective_halo+effective_patch, 
                                            effective_halo:effective_halo+effective_patch]
                    
                    # Accumulate
                    full_raw_mask[:, :, i_safe:i_safe+effective_patch, j_safe:j_safe+effective_patch] += center_mask
                    count_mask[:, :, i_safe:i_safe+effective_patch, j_safe:j_safe+effective_patch] += 1

            # Average overlapping predictions
            full_raw_mask = full_raw_mask / count_mask.clamp(min=1)
            binary_mask = (full_raw_mask > 0.5).float()

            # Step 2: Dilation
            mask_np = binary_mask.squeeze().cpu().numpy()
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilation_k, dilation_k))
            dilated_np = cv2.dilate(mask_np, kernel, iterations=1)
            full_dilated_mask = torch.from_numpy(dilated_np).unsqueeze(0).unsqueeze(0).to(DEVICE)

            # Step 3: Patch-based Inpainting with Context Halo
            final_output = torch.zeros_like(image_tensor)
            output_count = torch.zeros_like(image_tensor)
            
            # Pad dilated mask too
            mask_padded = F.pad(full_dilated_mask, (effective_halo, effective_halo, effective_halo, effective_halo), mode='reflect')

            for i in range(0, H, stride):
                for j in range(0, W, stride):
                    i_safe = min(i, H - effective_patch)
                    j_safe = min(j, W - effective_patch)
                    
                    # Extract large context patches
                    y_start = i_safe
                    x_start = j_safe
                    y_end = i_safe + effective_patch + (2 * effective_halo)
                    x_end = j_safe + effective_patch + (2 * effective_halo)
                    
                    img_large = img_padded[:, :, y_start:y_end, x_start:x_end]
                    mask_large = mask_padded[:, :, y_start:y_end, x_start:x_end]
                    
                    # Resize to model size
                    img_resized = T.Resize((224, 224), antialias=True)(img_large)
                    mask_resized = T.Resize((224, 224), antialias=True)(mask_large)
                    
                    # Prepare inpainting input
                    masked_img = img_resized * (1 - mask_resized)
                    inp_input = torch.cat([masked_img, mask_resized], dim=1)
                    
                    # Predict
                    pred_resized = self.inp_model(inp_input)
                    
                    # Resize back to context size
                    context_size = effective_patch + 2 * effective_halo
                    pred_large = T.Resize((context_size, context_size), antialias=True)(pred_resized)
                    img_large_result = img_large * (1 - mask_large) + pred_large * mask_large
                    
                    # Crop center
                    center_result = img_large_result[:, :, effective_halo:effective_halo+effective_patch, 
                                                     effective_halo:effective_halo+effective_patch]
                    
                    # Accumulate
                    final_output[:, :, i_safe:i_safe+effective_patch, j_safe:j_safe+effective_patch] += center_result
                    output_count[:, :, i_safe:i_safe+effective_patch, j_safe:j_safe+effective_patch] += 1
            
            final_output = final_output / output_count.clamp(min=1)
            
        return final_output, full_dilated_mask, binary_mask

In [ ]:
# Cell: Imports (Update to include these utilities)
def get_patches_with_halo(image_height, image_width, patch_size, halo_size):
    """Calculates coordinates for patching with overlap."""
    coords = []
    stride = patch_size - 2 * halo_size
    for y in range(0, image_height, stride):
        for x in range(0, image_width, stride):
            # Calculate patch bounds (with halo)
            y1 = max(0, y - halo_size)
            x1 = max(0, x - halo_size)
            y2 = min(image_height, y + patch_size - halo_size)
            x2 = min(image_width, x + patch_size - halo_size)
            
            # Adjust if patch hits the right/bottom edge
            if y2 == image_height: y1 = max(0, y2 - patch_size)
            if x2 == image_width:  x1 = max(0, x2 - patch_size)
            
            # Valid region to extract from the resulting patch
            v_y1 = y - y1
            v_x1 = x - x1
            v_y2 = v_y1 + stride if y + stride <= image_height else v_y1 + (image_height - y)
            v_x2 = v_x1 + stride if x + stride <= image_width else v_x1 + (image_width - x)
            
            coords.append({
                'crop': (y1, x1, y1 + patch_size, x1 + patch_size),
                'valid_src': (v_y1, v_x1, v_y2, v_x2),
                'valid_dst': (y, x, y + (v_y2 - v_y1), x + (v_x2 - v_x1))
            })
    return coords

In [13]:
import gradio as gr
import torch
import numpy as np
import cv2
from PIL import Image, ImageOps
import torchvision.transforms.v2 as T
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Assuming these are defined elsewhere in your code
# MEAN and STD for normalization
MEAN = torch.tensor([0.485, 0.456, 0.406])
STD = torch.tensor([0.229, 0.224, 0.225])

def denormalize(tensor):
    """Denormalize tensor for visualization"""
    mean = MEAN.view(3, 1, 1).to(tensor.device)
    std = STD.view(3, 1, 1).to(tensor.device)
    return tensor * std + mean


def get_patches_with_halo(H, W, patch_size, halo_size):
    """Generate patch coordinates with halo for context"""
    patches = []
    stride = patch_size
    
    for i in range(0, H, stride):
        for j in range(0, W, stride):
            # Core patch boundaries
            y1_core = i
            x1_core = j
            y2_core = min(i + patch_size, H)
            x2_core = min(j + patch_size, W)
            
            # Add halo (context)
            y1 = max(0, y1_core - halo_size)
            x1 = max(0, x1_core - halo_size)
            y2 = min(H, y2_core + halo_size)
            x2 = min(W, x2_core + halo_size)
            
            # Valid source region (where to extract from patch)
            vs_y1 = y1_core - y1
            vs_x1 = x1_core - x1
            vs_y2 = vs_y1 + (y2_core - y1_core)
            vs_x2 = vs_x1 + (x2_core - x1_core)
            
            # Valid destination region (where to place in output)
            vd_y1 = y1_core
            vd_x1 = x1_core
            vd_y2 = y2_core
            vd_x2 = x2_core
            
            patches.append({
                'crop': (y1, x1, y2, x2),
                'valid_src': (vs_y1, vs_x1, vs_y2, vs_x2),
                'valid_dst': (vd_y1, vd_x1, vd_y2, vd_x2)
            })
    
    return patches


class FenceRemovalPipeline:
    def __init__(self, seg_ckpt, inp_ckpt):
        self.seg_model = FenceSegmentationLightning.load_from_checkpoint(seg_ckpt).to(DEVICE).eval()
        self.inp_model = LightningModuleInpainting.load_from_checkpoint(inp_ckpt).to(DEVICE).eval()

    def process_simple(self, image_tensor, dilation_k=5, patch_size=224, halo_size=32):
        """
        Simple method: Processes the image using a patch-based approach with fixed halo.
        """
        _, _, H, W = image_tensor.shape
        final_output = image_tensor.clone()
        full_dilated_mask = torch.zeros((1, 1, H, W), device=DEVICE)
        full_raw_mask = torch.zeros((1, 1, H, W), device=DEVICE)
        
        # Calculate patch coordinates
        patch_coords = get_patches_with_halo(H, W, patch_size, halo_size)

        with torch.no_grad():
            # Step 1: Patch-based Segmentation
            for c in patch_coords:
                y1, x1, y2, x2 = c['crop']
                patch = image_tensor[:, :, y1:y2, x1:x2]
                
                # Resize patch to 224x224 for model
                patch_resized = T.Resize((224, 224), antialias=True)(patch)
                
                # Predict Mask
                seg_logits = self.seg_model(patch_resized)
                patch_raw_mask_resized = (torch.sigmoid(seg_logits) > 0.5).float()
                
                # Resize mask back to original patch size
                patch_h, patch_w = y2 - y1, x2 - x1
                patch_raw_mask = T.Resize((patch_h, patch_w), antialias=True)(patch_raw_mask_resized)
                
                # Assign valid part to full mask
                vs_y1, vs_x1, vs_y2, vs_x2 = c['valid_src']
                vd_y1, vd_x1, vd_y2, vd_x2 = c['valid_dst']
                full_raw_mask[:, :, vd_y1:vd_y2, vd_x1:vd_x2] = patch_raw_mask[:, :, vs_y1:vs_y2, vs_x1:vs_x2]

            # Step 2: Dilation
            mask_np = full_raw_mask.squeeze().cpu().numpy()
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilation_k, dilation_k))
            dilated_np = cv2.dilate(mask_np, kernel, iterations=1)
            full_dilated_mask = torch.from_numpy(dilated_np).unsqueeze(0).unsqueeze(0).to(DEVICE)

            # Step 3: Patch-based Inpainting
            for c in patch_coords:
                y1, x1, y2, x2 = c['crop']
                patch = image_tensor[:, :, y1:y2, x1:x2]
                
                # Predict Mask
                patch_resized = T.Resize((224, 224), antialias=True)(patch)
                seg_logits = self.seg_model(patch_resized)
                patch_raw_mask_resized = (torch.sigmoid(seg_logits) > 0.5).float()
                
                # Resize mask back to the CROP size (which includes halo)
                patch_h, patch_w = y2 - y1, x2 - x1
                patch_full_mask = T.Resize((patch_h, patch_w), antialias=True)(patch_raw_mask_resized)
                
                # CORRECTED ASSIGNMENT:
                # We must crop the 'valid' core out of the predicted patch 
                # before putting it into the full destination mask.
                vs_y1, vs_x1, vs_y2, vs_x2 = c['valid_src']
                vd_y1, vd_x1, vd_y2, vd_x2 = c['valid_dst']
                
                # Extract only the valid core from the patch that has a halo
                valid_mask_part = patch_full_mask[:, :, vs_y1:vs_y2, vs_x1:vs_x2]
                
                # Assign to the global mask
                full_raw_mask[:, :, vd_y1:vd_y2, vd_x1:vd_x2] = valid_mask_part
            
        return final_output, full_dilated_mask, full_raw_mask

    def process_complex(self, image_tensor, dilation_k=5, patch_size=224, halo_size=224):
        """
        Complex method: Processes with dynamic context halo and resizing.
        """
        _, _, H, W = image_tensor.shape
        
        # Adapt halo size to image dimensions
        max_halo_h = max(1, H - 1)
        max_halo_w = max(1, W - 1)
        effective_halo = min(halo_size, max_halo_h, max_halo_w)
        
        # If image is smaller than patch_size, adjust patch_size too
        effective_patch = min(patch_size, H, W)
        stride = effective_patch
        
        # Pad image to handle edges
        img_padded = F.pad(image_tensor, (effective_halo, effective_halo, effective_halo, effective_halo), mode='reflect')
        
        full_raw_mask = torch.zeros((1, 1, H, W), device=DEVICE)
        count_mask = torch.zeros((1, 1, H, W), device=DEVICE)

        with torch.no_grad():
            # Step 1: Patch-based Segmentation with Context Halo
            for i in range(0, H, stride):
                for j in range(0, W, stride):
                    i_safe = min(i, H - effective_patch)
                    j_safe = min(j, W - effective_patch)
                    
                    y_start = i_safe
                    x_start = j_safe
                    y_end = i_safe + effective_patch + (2 * effective_halo)
                    x_end = j_safe + effective_patch + (2 * effective_halo)
                    
                    large_patch = img_padded[:, :, y_start:y_end, x_start:x_end]
                    
                    # Resize to model input size
                    input_patch = T.Resize((224, 224), antialias=True)(large_patch)
                    
                    # Predict mask
                    seg_logits = self.seg_model(input_patch)
                    probs = torch.sigmoid(seg_logits)
                    
                    # Resize mask back to context size
                    context_size = effective_patch + 2 * effective_halo
                    mask_large = T.Resize((context_size, context_size), antialias=True)(probs)
                    
                    # Crop out center (removing halo)
                    center_mask = mask_large[:, :, effective_halo:effective_halo+effective_patch, 
                                            effective_halo:effective_halo+effective_patch]
                    
                    # Accumulate
                    full_raw_mask[:, :, i_safe:i_safe+effective_patch, j_safe:j_safe+effective_patch] += center_mask
                    count_mask[:, :, i_safe:i_safe+effective_patch, j_safe:j_safe+effective_patch] += 1

            # Average overlapping predictions
            full_raw_mask = full_raw_mask / count_mask.clamp(min=1)
            binary_mask = (full_raw_mask > 0.5).float()

            # Step 2: Dilation
            mask_np = binary_mask.squeeze().cpu().numpy()
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilation_k, dilation_k))
            dilated_np = cv2.dilate(mask_np, kernel, iterations=1)
            full_dilated_mask = torch.from_numpy(dilated_np).unsqueeze(0).unsqueeze(0).to(DEVICE)

            # Step 3: Patch-based Inpainting with Context Halo
            final_output = torch.zeros_like(image_tensor)
            output_count = torch.zeros_like(image_tensor)
            
            mask_padded = F.pad(full_dilated_mask, (effective_halo, effective_halo, effective_halo, effective_halo), mode='reflect')

            for i in range(0, H, stride):
                for j in range(0, W, stride):
                    i_safe = min(i, H - effective_patch)
                    j_safe = min(j, W - effective_patch)
                    
                    y_start = i_safe
                    x_start = j_safe
                    y_end = i_safe + effective_patch + (2 * effective_halo)
                    x_end = j_safe + effective_patch + (2 * effective_halo)
                    
                    img_large = img_padded[:, :, y_start:y_end, x_start:x_end]
                    mask_large = mask_padded[:, :, y_start:y_end, x_start:x_end]
                    
                    # Resize to model size
                    img_resized = T.Resize((224, 224), antialias=True)(img_large)
                    mask_resized = T.Resize((224, 224), antialias=True)(mask_large)
                    
                    # Prepare inpainting input
                    masked_img = img_resized * (1 - mask_resized)
                    inp_input = torch.cat([masked_img, mask_resized], dim=1)
                    
                    # Predict
                    pred_resized = self.inp_model(inp_input)
                    
                    # Resize back to context size
                    context_size = effective_patch + 2 * effective_halo
                    pred_large = T.Resize((context_size, context_size), antialias=True)(pred_resized)
                    img_large_result = img_large * (1 - mask_large) + pred_large * mask_large
                    
                    # Crop center
                    center_result = img_large_result[:, :, effective_halo:effective_halo+effective_patch, 
                                                     effective_halo:effective_halo+effective_patch]
                    
                    # Accumulate
                    final_output[:, :, i_safe:i_safe+effective_patch, j_safe:j_safe+effective_patch] += center_result
                    output_count[:, :, i_safe:i_safe+effective_patch, j_safe:j_safe+effective_patch] += 1
            
            final_output = final_output / output_count.clamp(min=1)
            
        return final_output, full_dilated_mask, binary_mask


# --- LOAD PIPELINE ---
pipeline = FenceRemovalPipeline(
    seg_ckpt="checkpoints/defence_model.ckpt",
    inp_ckpt="checkpoints/augmentation_mask_mix.ckpt"
)


# --- GRADIO FUNCTIONS ---
def auto_detect(input_dict, method, dilation):
    """Auto-detect fence mask using selected method"""
    # 1. Load and Fix Orientation (CRITICAL FIX)
    # We must bake the EXIF rotation into the pixels so the tensor matches the visual image
    raw_image = input_dict["background"]
    img_pil = ImageOps.exif_transpose(raw_image).convert("RGB")
    
    transform = T.Compose([
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=MEAN.tolist(), std=STD.tolist())
    ])
    img_tensor = transform(img_pil).unsqueeze(0).to(DEVICE)
    
    # Choose method
    if method == "Simple (Fast)":
        _, dilated_mask, _ = pipeline.process_simple(img_tensor, dilation_k=dilation)
    else:  # Complex (Better Quality)
        _, dilated_mask, _ = pipeline.process_complex(img_tensor, dilation_k=dilation)
    
    # Convert mask to RGBA for visualization
    # dilated_mask shape is (1, 1, H, W) -> squeeze -> (H, W)
    mask_np = (dilated_mask.squeeze().cpu().numpy() * 255).astype(np.uint8)
    
    # Create RGBA Mask
    rgba_mask = np.zeros((mask_np.shape[0], mask_np.shape[1], 4), dtype=np.uint8)
    rgba_mask[..., 0:3] = 255  # White color
    rgba_mask[..., 3] = mask_np  # Alpha channel

    mask_pil = Image.fromarray(rgba_mask)
    
    # 2. Safety Check: Ensure Mask matches Image size exactly
    # PIL size is (Width, Height), NumPy shape is (Height, Width)
    if mask_pil.size != img_pil.size:
        print(f"Resizing mask from {mask_pil.size} to {img_pil.size} to fix offset.")
        mask_pil = mask_pil.resize(img_pil.size, resample=Image.NEAREST)

    print(img_pil.size)
    print(mask_pil.size)
    
    return {
        "background": img_pil, # Return the EXIF-fixed image so the editor updates
        "layers": [mask_pil], 
        "composite": None
    }


def delete_fence(input_dict, method):
    """Remove fence using the manually edited mask"""
    # 1. Load and Fix Orientation (Consistency check)
    raw_image = input_dict["background"]
    img_pil = ImageOps.exif_transpose(raw_image).convert("RGB")
    
    # Handle case where user didn't run detection and mask is empty or None
    if not input_dict["layers"]:
        return img_pil
        
    mask_pil = input_dict["layers"][0].convert("L")
    
    # Ensure mask size matches image size before processing
    if mask_pil.size != img_pil.size:
         mask_pil = mask_pil.resize(img_pil.size, resample=Image.NEAREST)
    
    transform = T.Compose([
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=MEAN.tolist(), std=STD.tolist())
    ])
    img_tensor = transform(img_pil).unsqueeze(0).to(DEVICE)
    mask_tensor = (T.ToTensor()(mask_pil) > 0.5).float().unsqueeze(0).to(DEVICE)
    
    # Run inpainting with the user's mask
    _, _, H, W = img_tensor.shape
    final_output = img_tensor.clone()
    
    # Use appropriate patch configuration based on method
    if method == "Simple (Fast)":
        patch_coords = get_patches_with_halo(H, W, 224, 32)
    else:
        patch_coords = get_patches_with_halo(H, W, 224, 224)
    
    with torch.no_grad():
        for c in patch_coords:
            y1, x1, y2, x2 = c['crop']
            img_p = img_tensor[:, :, y1:y2, x1:x2]
            mask_p = mask_tensor[:, :, y1:y2, x1:x2]
            
            # Resize to 224x224 for model
            patch_h, patch_w = y2 - y1, x2 - x1
            img_p_resized = T.Resize((224, 224), antialias=True)(img_p)
            mask_p_resized = T.Resize((224, 224), antialias=True)(mask_p)
            
            # Prepare input
            masked_p = img_p_resized * (1 - mask_p_resized)
            inp_input = torch.cat([masked_p, mask_p_resized], dim=1)
            
            # Predict
            pred_p_resized = pipeline.inp_model(inp_input)
            
            # Resize back to original patch size
            pred_p = T.Resize((patch_h, patch_w), antialias=True)(pred_p_resized)
            
            # Reconstruct
            res_p = img_p * (1 - mask_p) + pred_p * mask_p
            
            vs_y1, vs_x1, vs_y2, vs_x2 = c['valid_src']
            vd_y1, vd_x1, vd_y2, vd_x2 = c['valid_dst']
            final_output[:, :, vd_y1:vd_y2, vd_x1:vd_x2] = res_p[:, :, vs_y1:vs_y2, vs_x1:vs_x2]

    res_pil = T.ToPILImage()(denormalize(final_output.squeeze(0)).cpu())
    return res_pil


# --- GRADIO UI ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🪚 DeFence: AI-Powered Fence Removal")
    gr.Markdown("""
    Remove fences from your images in 4 easy steps:
    1. **Upload** your image
    2. **Select** processing method and adjust dilation
    3. **Auto-detect** the fence (optionally refine with brush)
    4. **Delete** the fence to get your clean image
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            method_radio = gr.Radio(
                choices=["Simple (Fast)", "Complex (Better Quality)"],
                value="Simple (Fast)",
                label="Processing Method",
                info="Simple is faster, Complex provides better quality for difficult cases"
            )
            
            dilation_slider = gr.Slider(
                minimum=1,
                maximum=15,
                value=7,
                step=2,
                label="Mask Dilation",
                info="Increase to expand detected fence area"
            )
            
            detect_btn = gr.Button("🔍 Auto Detect Fence", variant="secondary", size="lg")
            delete_btn = gr.Button("✨ Delete Fence", variant="primary", size="lg")
            
            gr.Markdown("""
            ### 💡 Tips:
            - **Simple mode**: Best for most images, faster processing
            - **Complex mode**: Better for large images or complex fences
            - **Dilation**: Increase if fence edges remain visible
            - **Manual refinement**: Use the brush to adjust the mask
            """)
    
        with gr.Column(scale=2):
            with gr.Row():
                input_editor = gr.ImageEditor(
                    label="Image + Mask Editor",
                    type="pil",
                    sources=["upload"],
                    brush=gr.Brush(colors=["#FFFFFF"], default_size=20),
                    layers=True,
                    height=500
                )
                output_img = gr.Image(
                    label="Result", 
                    type="pil",
                    height=500
                )
    
    # Connect buttons to functions
    detect_btn.click(
        fn=auto_detect, 
        inputs=[input_editor, method_radio, dilation_slider], 
        outputs=input_editor
    )
    delete_btn.click(
        fn=delete_fence, 
        inputs=[input_editor, method_radio], 
        outputs=output_img
    )

if __name__ == "__main__":
    demo.launch()

/tmp/ipykernel_538355/3015881443.py:370: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


(700, 460)
(700, 460)
(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)


/home/alumno/py313ml/.venv/lib/python3.13/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


(1024, 1024)
(1024, 1024)


...

/home/alumno/py313ml/.venv/lib/python3.13/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


(1024, 1024)
(1024, 1024)
(1024, 1024)
(1024, 1024)


/home/alumno/py313ml/.venv/lib/python3.13/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


(3264, 1840)
(3264, 1840)
(3264, 1840)
(3264, 1840)


/home/alumno/py313ml/.venv/lib/python3.13/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(
